In [3]:
import numpy as np
import xarray
import os
from matplotlib import pyplot as plt
import scipy.signal as scipy_signal
import scipy.stats as scipy_stats
from geopy import distance as geo_dist
import cmocean
import pandas
import proplot
import cartopy.feature as cfeature

In [ ]:
cyclone_dataframe.columns

In [6]:
track_file_path = '/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC'
olive_ridley_dir = 'tu117'
flatback_dir     = 'tu120'


olive_ridley_files = os.listdir(os.path.join(track_file_path,olive_ridley_dir))
olive_ridley_files_list = []

for i_file in olive_ridley_files:
    
    if i_file.endswith('_hr1_prof.nc'):
        olive_ridley_files_list.append(os.path.join(track_file_path,olive_ridley_dir,i_file))
        

flatback_files = os.listdir(os.path.join(track_file_path,flatback_dir))
flatback_files_list = []

for i_file in flatback_files:
    
    if i_file.endswith('_hr1_prof.nc'):
        flatback_files_list.append(os.path.join(track_file_path,flatback_dir,i_file))



nws_track_file_path = '/oa-decadal-climate/work/observations/Anibos_Turtles/NW_Deployment'
nws_turtle_files_list = os.listdir(nws_track_file_path)
nws_turtle_files = []

for i_file in nws_turtle_files_list:
    
    if i_file.endswith('_lr1_prof.nc'):
        nws_turtle_files.append(os.path.join(nws_track_file_path,i_file))

turtle_files = olive_ridley_files_list + flatback_files_list + nws_turtle_files



#track_data_frame = pandas.read_csv(track_file_name)

In [10]:
first_pass = True


turtle_dataframe  = {} #{'id':[],'lat':[],'lon':[],'time':[]}


for i_turtle in turtle_files:
    turtle_ds = xarray.open_dataset(i_turtle,use_cftime=True)
    #print(turtle_ds)
    time = xarray.CFTimeIndex(turtle_ds['JULD'].values).to_datetimeindex()
    
    if first_pass:
        min_time = time.min()
        max_time = time.max()
        
        
        turtle_dataframe['id'] = turtle_ds['PLATFORM_NUMBER'].values.astype('int')
        turtle_dataframe['lat'] = turtle_ds['LATITUDE'].values
        turtle_dataframe['lon'] = turtle_ds['LONGITUDE'].values
        turtle_dataframe['date'] = time


        
        first_pass = False
    else:
        if time.min()<min_time:
            min_time = time.min()
        if time.max()>max_time:
            max_time = time.max()
        
        if 'PLATFORM_NUMBER' in turtle_ds.keys():
            
            turtle_dataframe['id']   = np.concatenate([turtle_dataframe['id'], turtle_ds['PLATFORM_NUMBER'].values.astype('int')])
            turtle_dataframe['lat']  = np.concatenate([turtle_dataframe['lat'], turtle_ds['LATITUDE'].values])
            turtle_dataframe['lon']  = np.concatenate([turtle_dataframe['lon'], turtle_ds['LONGITUDE'].values])
            turtle_dataframe['date'] = np.concatenate([turtle_dataframe['date'], time])
        else:
            turtle_dataframe['id']   = np.concatenate([turtle_dataframe['id'], ])

turtle_dataframe = pandas.DataFrame(data=turtle_dataframe)

/tube1/cha674/Anaconda_Install/miniconda/envs/py3/lib/python3.7/site-packages/ipykernel_launcher.py:10: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'julian', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  # Remove the CWD from sys.path while we load stuff.


DatetimeIndex(['2013-04-25 23:09:22.500000', '2013-04-26 23:20:37.500000',
               '2013-04-27 05:00:56.250000',        '2013-04-27 17:48:45',
               '2013-04-28 05:00:56.250000', '2013-04-29 05:20:37.500000',
               '2013-04-30 05:09:22.500000', '2013-05-04 11:20:37.500000',
               '2013-05-04 20:20:37.500000', '2013-05-05 03:39:22.500000',
               '2013-05-05 09:39:22.500000', '2013-05-05 15:39:22.500000',
                      '2013-05-05 21:11:15',        '2013-05-06 03:11:15',
               '2013-05-06 08:09:22.500000', '2013-05-07 15:19:41.250000',
                      '2013-05-09 04:18:45'],
              dtype='datetime64[ns]', freq=None)
DatetimeIndex([       '2013-02-11 19:41:15',        '2013-02-12 01:30:00',
                      '2013-02-12 07:18:45', '2013-02-13 06:30:56.250000',
                      '2013-02-13 19:18:45',        '2013-02-14 16:18:45',
                      '2013-02-14 23:48:45', '2013-02-15 03:50:37.500000',
     

In [11]:
turtle_dataframe['id'].unique()

array([103302, 103332, 103416, 103410, 103326, 103218, 103440, 103398,
       103446, 103362, 103434, 103404, 103284, 103422, 103392, 103308,
       103290, 103380, 103428, 103296, 103242, 103230, 105246, 106686,
       103272, 103266, 103236, 103248, 103254, 103368, 103314, 103356,
       103224, 105252, 103374, 103260, 103206, 103278, 103464])

In [13]:
cyclone_dataframe = pandas.read_csv('./IDCKMSTM0S.csv')

cyclone_time = []
cyclone_lat  = []
cyclone_lon  = []
cyclone_id   = []
cyclone_name = []
central_pressure = []
cyclone_radius = []
cyclone_gale_force_wind_radius = []
surface_code = []

counter = 0
for i_time in cyclone_dataframe['TM']:
    
    if isinstance(i_time, str) and not i_time.isspace():
        cyclone_time.append(pandas.to_datetime(i_time))
        cyclone_lat.append(float(cyclone_dataframe['LAT'].loc[counter]))
        cyclone_lon.append(float(cyclone_dataframe['LON'].loc[counter]))
        cyclone_name.append(str(cyclone_dataframe['NAME'].loc[counter]))
        cyclone_id.append(str(cyclone_dataframe['DISTURBANCE_ID'].loc[counter]))
        
        if not cyclone_dataframe['CENTRAL_PRES'].loc[counter].isspace():
            
            central_pressure.append(cyclone_dataframe['CENTRAL_PRES'].loc[counter])
        else:
            central_pressure.append('nan')

        if not cyclone_dataframe['MN_RADIUS_OUTER_ISOBAR'].loc[counter].isspace():
            cyclone_radius.append(float(cyclone_dataframe['MN_RADIUS_OUTER_ISOBAR'].loc[counter]))
        else:
            cyclone_radius.append('nan')
        
        if not cyclone_dataframe['MN_RADIUS_GF_WIND'].loc[counter].isspace():
            cyclone_gale_force_wind_radius.append(float(cyclone_dataframe['MN_RADIUS_GF_WIND'].loc[counter]))
        else:
            cyclone_gale_force_wind_radius.append('nan')
        
        if not cyclone_dataframe['SURFACE_CODE'].loc[counter].isspace():
            surface_code.append(float(cyclone_dataframe['SURFACE_CODE'].loc[counter]))
        else:
            surface_code.append(np.nan)


    else:
         print('Missing date: ', counter)
         #print(i_time)
    counter = counter +1
    
cyclone_time = pandas.to_datetime(cyclone_time)
cyclone_lat = np.asarray(cyclone_lat).astype('float32')
cyclone_lon = np.asarray(cyclone_lon).astype('float32')
central_pressure = np.asarray(central_pressure).astype('float32')
cyclone_radius  = np.asarray(cyclone_radius).astype('float32')
cyclone_gale_force_wind_radius  = np.asarray(cyclone_gale_force_wind_radius).astype('float32')
surface_code = np.asarray(surface_code).astype('float32')

Missing date:  6570
Missing date:  6571
Missing date:  8533
Missing date:  8534
Missing date:  9627
Missing date:  9628
Missing date:  10542
Missing date:  10543
Missing date:  12253
Missing date:  12254
Missing date:  15711
Missing date:  15712
Missing date:  24255
Missing date:  24256
Missing date:  24772
Missing date:  24773
Missing date:  27830
Missing date:  27831
Missing date:  28604
Missing date:  28605
Missing date:  29009
Missing date:  29010
Missing date:  29107
Missing date:  29108
Missing date:  29430
Missing date:  29431
Missing date:  30283
Missing date:  30284


In [65]:
cyclone_dataframe['SURFACE_CODE'].loc[counter]

' '

In [ ]:
plt.plot(cyclone_radius)

In [ ]:
idx_time = np.nonzero(np.logical_and(cyclone_time>turtle_dataframe['date'].min(),
                                     cyclone_time<turtle_dataframe['date'].max()))[0]

In [ ]:
plt.scatter(cyclone_lon[idx_time],cyclone_lat[idx_time])
plt.scatter(turtle_dataframe['lon'],turtle_dataframe['lat'])

In [15]:
from cartopy.io.shapereader import Reader
from sklearn.metrics.pairwise import haversine_distances

EARTH_RADIUS = 6371000
aus_eez_regions= Reader('./au_eez_pol_april2022/au_eez_pol_april2022.shp')
eez_boundary = aus_eez_regions.geometries() # get all the polygons
aus_eez_polygons = list(eez_boundary)
aus_main = aus_eez_polygons[3]

turtle_storm_dist = {}
print('---------------------')
for i_turtle in turtle_files:
    
    print(i_turtle)
    current_turtle = xarray.open_dataset(i_turtle,decode_times=True,use_cftime=True)
    
    #current_id = np.unique(current_turtle['PLATFORM_NUMBER']).astype('int')
    
    n_profiles = current_turtle['JULD'].size
    
    
    turtle_lons = current_turtle['LONGITUDE'].values
    turtle_lats = current_turtle['LATITUDE'].values
    turtle_time = xarray.CFTimeIndex(current_turtle['JULD'].values).to_datetimeindex()
    
    dist_to_storm  = []
    storm_name     = []
    storm_pressure = []
    storm_radius   = []
    storm_time     = []
    storm_gf_winds_radius = []
    storm_surface_code = []
    
    for i_profile in range(0,n_profiles):
        turtle_date = turtle_time[i_profile].date()
        idx_time = np.nonzero(cyclone_time.date==turtle_date)[0]
        if idx_time.size !=0:
            #print('Found one')
            
            closest_idx = np.argmin(np.abs(turtle_time[i_profile]-cyclone_time[idx_time]))
            
            dist_cyclone_turtle = haversine_distances( [ [np.deg2rad(cyclone_lat[idx_time][closest_idx]), np.deg2rad(cyclone_lon[idx_time][closest_idx]) ],
                                                         [np.deg2rad(turtle_lats[i_profile]), np.deg2rad(turtle_lons[i_profile]) ] ] ) * EARTH_RADIUS
            
            
            
            storm_time.append(turtle_time[i_profile])
            dist_to_storm.append(dist_cyclone_turtle[0,1])
            storm_radius.append(  cyclone_radius[idx_time[closest_idx]])
            storm_gf_winds_radius.append(  cyclone_gale_force_wind_radius[idx_time[closest_idx]])

            storm_pressure.append(central_pressure[idx_time[closest_idx]])
            
            storm_surface_code.append(surface_code[idx_time[closest_idx]])
            
            current_storm_name = cyclone_name[idx_time[closest_idx]].strip()
            current_storm_id   = cyclone_id[idx_time[closest_idx]].strip()

            
            if current_storm_name == 'noname':
                storm_name.append(current_storm_id)
            else:
                storm_name.append(current_storm_name)
           
    turtle_storm_dist[os.path.basename(i_turtle)] = {}
    turtle_storm_dist[os.path.basename(i_turtle)]['distance_to_storm'] = np.asarray(dist_to_storm)
    turtle_storm_dist[os.path.basename(i_turtle)]['storm_name']        = storm_name
    
    turtle_storm_dist[os.path.basename(i_turtle)]['storm_radius']       = np.asarray(storm_radius)
    turtle_storm_dist[os.path.basename(i_turtle)]['storm_pressure']     = np.asarray(storm_pressure)
    turtle_storm_dist[os.path.basename(i_turtle)]['storm_time']         = np.asarray(storm_time)
    turtle_storm_dist[os.path.basename(i_turtle)]['storm_wind_radius']   = np.asarray(storm_gf_winds_radius)
    turtle_storm_dist[os.path.basename(i_turtle)]['storm_surface_code']   = np.asarray(storm_surface_code)


    current_turtle.close()
    


---------------------
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-672-22_hr1_prof.nc


/tube1/cha674/Anaconda_Install/miniconda/envs/py3/lib/python3.7/site-packages/ipykernel_launcher.py:24: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'julian', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.


/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-671-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-675-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-686-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-617-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-684-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-688-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-676-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-679-22_hr1_prof.nc
/oa-decadal-climate/work/observations/Anibos_Turtles/IMOS_turtle_profiles_QC/tu117/tu117-614-22_hr1_prof.nc
/oa-decadal-climate/work/obs

In [16]:
M_TO_KM = 1.0e-3

storm_metrics_to_save = {'turtle_id': [], 'date_storm_passage': [], 'minimum_distance_km': [], 'storm_name': []  }
        
for i_turtle in turtle_storm_dist:
    distance_to_storm = M_TO_KM * turtle_storm_dist[i_turtle]['distance_to_storm'] 
    storm_radius      = turtle_storm_dist[i_turtle]['storm_radius']
    storm_name        = turtle_storm_dist[i_turtle]['storm_name']
    storm_wind_radius    = turtle_storm_dist[i_turtle]['storm_wind_radius']
    storm_passage_time   = turtle_storm_dist[i_turtle]['storm_time']
    storm_surface_code   = turtle_storm_dist[i_turtle]['storm_surface_code']

    if len(distance_to_storm) !=0:
        
        
        idx_turtle_in_radius = np.nonzero(distance_to_storm < storm_radius)[0]
        
        if len(idx_turtle_in_radius) !=0:
            unique_storm_names = []
            for i_timestamp in idx_turtle_in_radius:
                if len(unique_storm_names) ==0:
                    unique_storm_names.append(storm_name[i_timestamp])
                elif unique_storm_names[-1] != storm_name[i_timestamp]:
                    unique_storm_names.append(storm_name[i_timestamp])
        
            for i_storm in unique_storm_names:
                storm_idx = [i for i, e in enumerate(storm_name) if e == i_storm]
                min_distance_idx = np.argmin(distance_to_storm[storm_idx])
                
                over_land = storm_surface_code[storm_idx][min_distance_idx] ==2.0
                if not over_land:
                    print(i_turtle,i_storm, distance_to_storm[storm_idx][min_distance_idx],
                      storm_passage_time[storm_idx][min_distance_idx],storm_surface_code[storm_idx][min_distance_idx])
                    
                    storm_metrics_to_save['turtle_id'].append(i_turtle)
                    storm_metrics_to_save['storm_name'].append(i_storm)
                    storm_metrics_to_save['minimum_distance_km'].append(distance_to_storm[storm_idx][min_distance_idx])
                    storm_metrics_to_save['date_storm_passage'].append(storm_passage_time[storm_idx][min_distance_idx])

                    

    
    

tu120-660-22_hr1_prof.nc AU202223_16U 312.2235704597376 2023-03-04 22:15:00.000003 4.0
tu120-619-22_hr1_prof.nc AU202223_16U 99.06931922539759 2023-02-27 01:19:59.999996 1.0
tu120-619-22_hr1_prof.nc Ilsa 62.368129379578924 2023-04-10 00:30:00.000003 1.0
tu120-668-22_hr1_prof.nc AU202223_16U 378.0020475959806 2023-03-05 07:35:00.000002 1.0
tu120-615-22_hr1_prof.nc Ellie 19.061553483418187 2022-12-21 20:09:59.999998 1.0
tu120-658-22_hr1_prof.nc AU202223_16U 531.9349241501465 2023-03-04 21:59:59.999997 4.0
tu120-665-I2C-22_hr1_prof.nc AU202223_16U 171.7883502761787 2023-02-26 08:04:59.999996 1.0
tu78-609-12_lr1_prof.nc Rusty 555.2054629783403 2013-02-26 00:30:56.250000 1.0
tu78-610-12_lr1_prof.nc Rusty 46.85591101396737 2013-02-27 06:11:15 1.0
tu78-611-12_lr1_prof.nc Christine 104.01200013660981 2013-12-30 11:40:18.750000 1.0
tu78-640-12_lr1_prof.nc Rusty 202.58169302651459 2013-02-25 21:11:15 1.0
tu78-642-12_lr1_prof.nc Rusty 290.82642085715173 2013-02-22 05:00:56.250000 1.0
tu78-642-12_

In [17]:
storm_turtle_interaction_dataframe = pandas.DataFrame.from_dict(storm_metrics_to_save)

In [18]:
storm_turtle_interaction_dataframe.to_csv('./turtle_storm_interaction.csv')

In [19]:
storm_turtle_interaction_dataframe

,turtle_id,date_storm_passage,minimum_distance_km,storm_name
0,tu120-660-22_hr1_prof.nc,2023-03-04 22:15:00.000003,312.223570,AU202223_16U
1,tu120-619-22_hr1_prof.nc,2023-02-27 01:19:59.999996,99.069319,AU202223_16U
2,tu120-619-22_hr1_prof.nc,2023-04-10 00:30:00.000003,62.368129,Ilsa
3,tu120-668-22_hr1_prof.nc,2023-03-05 07:35:00.000002,378.002048,AU202223_16U
4,tu120-615-22_hr1_prof.nc,2022-12-21 20:09:59.999998,19.061553,Ellie
5,tu120-658-22_hr1_prof.nc,2023-03-04 21:59:59.999997,531.934924,AU202223_16U
6,tu120-665-I2C-22_hr1_prof.nc,2023-02-26 08:04:59.999996,171.788350,AU202223_16U
7,tu78-609-12_lr1_prof.nc,2013-02-26 00:30:56.250000,555.205463,Rusty
8,tu78-610-12_lr1_prof.nc,2013-02-27 06:11:15.000000,46.855911,Rusty
9,tu78-611-12_lr1_prof.nc,2013-12-30 11:40:18.750000,104.012000,Christine


In [ ]:
turtle_storm_dist[i_turtle]['storm_name']

In [ ]:
distance_threshold = 200e3

    
    storm_name = np.unique(turtle_storm_dist[i_turtle]['storm_name'])
    storm_index = [i for i, e in enumerate(storm_name) if e == 'AU202223_16U']

    for i_storm in storm_name:
        storm_index = [i for i, e in enumerate(storm_name) if e == i_storm]
        idx_current_storm = np.nonzero(turtle_storm_dist[i_turtle]['storm_name']==i_storm)[0]
    
    dsada
    
    
    turtle_near_storm_idx = np.nonzero(turtle_storm_dist[i_turtle]['distance_to_storm']*1.0e-3<turtle_storm_dist[i_turtle]['storm_radius']*0.5)[0]
    
    if len(turtle_near_storm_idx) !=0:
        print(i_turtle)
        print(np.nanmin(turtle_storm_dist[i_turtle]['distance_to_storm'])*1.0e-3)
        print(turtle_storm_dist[i_turtle]['storm_time'] )
        print('found one!')
    
    

In [ ]:
plt.plot(turtle_storm_dist[i_turtle]['storm_time'],turtle_storm_dist[i_turtle]['storm_radius'])
plt.plot(turtle_storm_dist[i_turtle]['storm_time'],turtle_storm_dist[i_turtle]['distance_to_storm']*1.0e-3)

In [ ]:
np.nanmin(dist_to_storm[isla_index])/1000pandas.to_datetime(turtle_time)[isla_index]

In [ ]:
isla_index = [i for i, e in enumerate(storm_name) if e == 'AU202223_16U']


In [ ]:
#plt.scatter(cyclone_lon[idx_time],cyclone_lat[idx_time])
plt.scatter(turtle_lons[isla_index],turtle_lats[isla_index])

In [ ]:
idx_min_dist = np.nanargmin(dist_to_storm)

In [ ]:
turtle_time[idx_min_dist]

In [ ]:
cyclone_time[idx_time][closest_idx]

In [ ]:
turtle_time[i_profile]